# 12 — Interview Q&A: Async Programming, Streams & Express

MCQs and detailed Q&A covering:
- Asynchronous Programming (Callbacks, Promises, async/await)
- EventEmitter & Streams
- Express.js & REST APIs

---

# PART A — Multiple Choice Questions

---

## Section 1: Async Programming MCQs

### MCQ 1
**What is the Error-First Callback pattern?**

A) The callback receives the result first, then the error  
B) The callback's first argument is always the error (or null if no error)  
C) The callback only receives errors, never results  
D) Errors are thrown instead of being passed to callbacks  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) The callback's first argument is always the error (or null if no error)**

This is the Node.js convention: `callback(err, result)`. On success: `callback(null, data)`. On failure: `callback(new Error('msg'))`. Always check `if (err)` first and return early.
</details>

### MCQ 2
**What are the three states of a Promise?**

A) `open`, `closed`, `error`  
B) `pending`, `fulfilled`, `rejected`  
C) `waiting`, `resolved`, `failed`  
D) `created`, `running`, `completed`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `pending`, `fulfilled`, `rejected`**

A Promise starts as **pending**. It either transitions to **fulfilled** (success, triggers `.then()`) or **rejected** (failure, triggers `.catch()`). Once settled (fulfilled or rejected), it cannot change state again.
</details>

### MCQ 3
**What does an `async` function always return?**

A) `undefined`  
B) The raw return value  
C) A Promise  
D) An EventEmitter  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) A Promise**

An `async` function always returns a Promise. If you `return 42`, it returns `Promise.resolve(42)`. If you throw, it returns a rejected Promise. This is why you can always `.then()` or `await` an async function.
</details>

### MCQ 4 — Output Prediction
**What is the output?**

```javascript
async function foo() {
    return 'hello';
}

const result = foo();
console.log(result);
```

A) `'hello'`  
B) `Promise { 'hello' }`  
C) `undefined`  
D) Throws an error  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `Promise { 'hello' }`**

Since `foo()` is async, it returns a Promise. Without `await`, `result` holds the Promise object, NOT the resolved value `'hello'`. To get the value, use `await foo()` inside another async function, or `foo().then(val => ...)`.
</details>

In [ ]:
// Verify MCQ 4
async function foo() {
    return 'hello';
}
const result = foo();
console.log(result);
console.log('Type:', typeof result, '| Is Promise:', result instanceof Promise);

### MCQ 5
**Which `Promise` combinator rejects immediately if ANY promise rejects?**

A) `Promise.allSettled()`  
B) `Promise.any()`  
C) `Promise.all()`  
D) `Promise.race()`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) `Promise.all()`**

- `Promise.all()` — resolves when ALL fulfill, rejects when ANY rejects (fail-fast)
- `Promise.allSettled()` — waits for ALL to settle, never rejects
- `Promise.race()` — settles with the FIRST to settle (fulfill or reject)
- `Promise.any()` — resolves with the FIRST to fulfill, rejects only if ALL reject
</details>

### MCQ 6 — Output Prediction
**What is the output?**

```javascript
const arr = [1, 2, 3];

arr.forEach(async (num) => {
    await new Promise(r => setTimeout(r, 100));
    console.log(num);
});

console.log('Done');
```

A) 1, 2, 3, Done  
B) Done, 1, 2, 3  
C) Done (then 1, 2, 3 after 100ms — all at once)  
D) 1, Done, 2, 3  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Done (then 1, 2, 3 after 100ms — all at once)**

`forEach` does NOT wait for async callbacks. It fires all three callbacks immediately (they each start their own `await`), then `console.log('Done')` runs. After 100ms, all three resolve roughly simultaneously and print 1, 2, 3.

**Fix:** Use `for...of` for sequential, or `Promise.all(arr.map(...))` for parallel-with-await.
</details>

In [ ]:
// Verify MCQ 6
const arr = [1, 2, 3];
arr.forEach(async (num) => {
    await new Promise(r => setTimeout(r, 100));
    console.log(num);
});
console.log('Done');

### MCQ 7
**What tool converts a callback-based function to a Promise-based one?**

A) `JSON.stringify()`  
B) `util.promisify()`  
C) `Buffer.from()`  
D) `EventEmitter.once()`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `util.promisify()`**

```javascript
const { promisify } = require('util');
const readFile = promisify(fs.readFile);
const data = await readFile('file.txt', 'utf8');
```

It works on functions that follow the error-first callback convention `(err, result) => ...`. Even better, use `fs.promises` or `require('fs/promises')` for the fs module.
</details>

### MCQ 8 — Output Prediction
**What is the output?**

```javascript
console.log('1');

setTimeout(() => console.log('2'), 10);
setTimeout(() => console.log('3'), 0);

Promise.resolve()
    .then(() => console.log('4'))
    .then(() => console.log('5'));

console.log('6');
```

A) 1, 6, 4, 5, 3, 2  
B) 1, 6, 3, 4, 5, 2  
C) 1, 4, 5, 6, 3, 2  
D) 1, 6, 4, 3, 5, 2  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) 1, 6, 4, 5, 3, 2**

1. `1` — sync
2. `6` — sync
3. `4` — microtask (Promise.then), runs before any macrotask
4. `5` — chained microtask, runs after `4`
5. `3` — macrotask (setTimeout 0ms)
6. `2` — macrotask (setTimeout 10ms)
</details>

In [ ]:
// Verify MCQ 8
console.log('1');
setTimeout(() => console.log('2'), 10);
setTimeout(() => console.log('3'), 0);
Promise.resolve()
    .then(() => console.log('4'))
    .then(() => console.log('5'));
console.log('6');

---
## Section 2: EventEmitter & Streams MCQs

### MCQ 9
**What happens if you emit an `'error'` event on an EventEmitter with no error listener?**

A) The error is silently ignored  
B) Node.js logs a warning but continues  
C) Node.js throws the error and crashes the process  
D) The error is stored in a queue for later handling  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Node.js throws the error and crashes the process**

The `'error'` event is special in Node.js. If there is no registered listener for it, Node.js throws the error as an unhandled exception, which crashes the process. Always add `.on('error', handler)` to your EventEmitters.
</details>

### MCQ 10
**What are the 4 types of Streams in Node.js?**

A) Input, Output, Bidirectional, Filter  
B) Readable, Writable, Duplex, Transform  
C) Read, Write, Pipe, Buffer  
D) Source, Sink, Channel, Processor  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Readable, Writable, Duplex, Transform**

- **Readable** — source of data (e.g., `fs.createReadStream()`)
- **Writable** — destination for data (e.g., `fs.createWriteStream()`)
- **Duplex** — both readable and writable (e.g., `net.Socket`)
- **Transform** — Duplex that modifies data passing through (e.g., `zlib.createGzip()`)
</details>

### MCQ 11
**What is backpressure in the context of Node.js streams?**

A) The pressure on the CPU from processing data  
B) When a readable stream produces data faster than the writable stream can consume it  
C) When the event loop is overloaded  
D) Memory pressure from too many open file handles  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) When a readable stream produces data faster than the writable stream can consume it**

Backpressure occurs when the write buffer fills up. `.write()` returns `false` to signal the producer to pause. The `'drain'` event fires when the buffer is ready for more data. `pipeline()` handles backpressure automatically.
</details>

### MCQ 12
**Why is `pipeline()` preferred over `.pipe()`?**

A) `pipeline()` is faster  
B) `pipeline()` handles errors automatically and cleans up streams on failure  
C) `.pipe()` doesn't support Transform streams  
D) `pipeline()` can only connect two streams  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `pipeline()` handles errors automatically and cleans up streams on failure**

`.pipe()` can silently swallow errors and leak resources (streams aren't properly destroyed). `pipeline()` propagates errors to a callback, auto-destroys all streams in the chain on failure, and supports a callback/Promise for completion notification.
</details>

### MCQ 13
**What does `emitter.once('event', handler)` do?**

A) Registers a handler that fires for every event emission  
B) Registers a handler that fires only once, then auto-removes itself  
C) Fires the handler immediately once  
D) Throws an error if the event has already been emitted  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Registers a handler that fires only once, then auto-removes itself**

`.once()` is useful for one-time events like `'connect'`, `'ready'`, or `'drain'`. After the first emission, the listener is automatically removed — subsequent emissions of the same event don't trigger it.
</details>

---
## Section 3: Express & REST API MCQs

### MCQ 14
**What is middleware in Express?**

A) A database connection layer  
B) Functions with access to `req`, `res`, and `next` that execute in a pipeline  
C) The routing engine of Express  
D) A template engine for rendering HTML  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Functions with access to `req`, `res`, and `next` that execute in a pipeline**

Middleware functions can: modify `req`/`res`, end the request-response cycle, or call `next()` to pass control to the next middleware. Express is essentially a series of middleware function calls. If you don't call `next()` or send a response, the request hangs.
</details>

### MCQ 15
**How does Express distinguish error-handling middleware from regular middleware?**

A) By the middleware name  
B) By having exactly 4 parameters: `(err, req, res, next)`  
C) By placing it before all other middleware  
D) By using `app.error()` instead of `app.use()`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) By having exactly 4 parameters: `(err, req, res, next)`**

Express checks the function's `.length` property (number of parameters). If it's 4, Express treats it as error-handling middleware. It's triggered when `next(error)` is called from any preceding middleware. Error middleware should be defined LAST.
</details>

### MCQ 16
**What is the difference between `app.use('/api')` and `app.get('/api')`?**

A) No difference  
B) `app.use` matches ALL methods and prefix paths; `app.get` matches only GET with exact path  
C) `app.get` is faster  
D) `app.use` only works with middleware, not route handlers  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `app.use` matches ALL methods and prefix paths; `app.get` matches only GET with exact path**

`app.use('/api')` matches GET, POST, PUT, DELETE, etc., AND matches `/api`, `/api/users`, `/api/anything`. `app.get('/api')` matches ONLY GET requests to exactly `/api` (not `/api/users`). This is why `app.use()` is used for middleware and `app.get()` for route handlers.
</details>

### MCQ 17
**What is the correct HTTP status code for a successfully created resource?**

A) 200 OK  
B) 201 Created  
C) 204 No Content  
D) 202 Accepted  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) 201 Created**

Status code cheat sheet:
- **200** — Successful GET, PUT, PATCH
- **201** — Successful POST (resource created)
- **204** — Successful DELETE (no content to return)
- **202** — Request accepted for async processing (not completed yet)
</details>

### MCQ 18
**What is the difference between HTTP PUT and PATCH?**

A) PUT creates, PATCH deletes  
B) PUT replaces the entire resource, PATCH applies a partial update  
C) PUT is idempotent, PATCH is not  
D) There is no difference  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) PUT replaces the entire resource, PATCH applies a partial update**

PUT: You must send ALL fields — missing fields are set to null/default. PATCH: You send only the fields you want to change. Both are idempotent (option C is false — PATCH is also idempotent in practice). Example: to change only a user's name, use PATCH with `{ name: 'New Name' }`, not PUT with all user fields.
</details>

### MCQ 19
**What does the `express.json()` middleware do?**

A) Sends JSON responses  
B) Validates JSON schemas  
C) Parses incoming JSON request bodies and makes them available on `req.body`  
D) Converts all responses to JSON format  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Parses incoming JSON request bodies and makes them available on `req.body`**

Without `express.json()`, `req.body` is `undefined`. This built-in middleware reads the raw request body, parses it as JSON, and assigns the result to `req.body`. It only activates when the `Content-Type` header is `application/json`.
</details>

### MCQ 20
**What is the difference between `401 Unauthorized` and `403 Forbidden`?**

A) They mean the same thing  
B) 401 means not authenticated (who are you?), 403 means not authorized (you can't do this)  
C) 401 is for server errors, 403 is for client errors  
D) 401 means resource not found, 403 means rate limited  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) 401 means not authenticated (who are you?), 403 means not authorized (you can't do this)**

- **401 Unauthorized** — The request lacks valid credentials. The client should authenticate (log in) and retry.
- **403 Forbidden** — The client is authenticated but does NOT have permission to access this resource. Re-authenticating won't help.

Example: A regular user trying to access an admin endpoint gets 403, not 401 (they are logged in, just not allowed).
</details>

---
## Section 4: Additional Async, Streams & Express MCQs


### MCQ 21
**What does `Promise.resolve(Promise.resolve(42))` return?**

A) `Promise<42>` — nested Promises are automatically unwrapped  
B) `42` (the raw value)  
C) `Promise<Promise<42>>` (a nested Promise)  
D) `undefined`

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) `Promise<42>` — nested Promises are automatically unwrapped**

`Promise.resolve()` unwraps "thenables" (objects with a `.then` method). When you pass a Promise to `Promise.resolve()`, it returns a Promise that adopts the state of the inner Promise. So `Promise.resolve(Promise.resolve(42))` resolves to `42`, not a nested Promise.
</details>

### MCQ 22 — Output Prediction
**What is the output?**

```javascript
async function foo() {
    return await 42;
}
foo().then(console.log);
console.log('hello');
```

A) hello, 42  
B) 42, hello  
C) hello  
D) 42

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) hello, 42**

The async function returns a Promise. `foo().then(console.log)` schedules a microtask to run when the Promise resolves. `console.log('hello')` runs synchronously first. Then the microtask runs and logs `42`. Microtasks run after the current synchronous code but before the next macrotask.
</details>

In [ ]:
// Verify MCQ 22
async function foo() {
    return await 42;
}
foo().then(console.log);
console.log('hello');

### MCQ 23
**What is the highWaterMark in a Node.js stream?**

A) The maximum file size the stream can handle  
B) The internal buffer size threshold that controls when to stop reading/signal backpressure (default 16KB for object mode or 16384 bytes)  
C) The number of concurrent pipe connections allowed  
D) The timeout in milliseconds before the stream closes

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) The internal buffer size threshold that controls when to stop reading/signal backpressure (default 16KB for object mode or 16384 bytes)**

highWaterMark specifies the maximum number of bytes (or objects in object mode) to buffer before the stream signals backpressure. When the buffer reaches this threshold, `readable.read()` returns `null` and the writable's `.write()` returns `false`, signaling the producer to slow down.
</details>

### MCQ 24
**What does `stream.destroy()` do?**

A) Pauses the stream temporarily  
B) Immediately closes the stream and releases resources, optionally emitting an error event  
C) Flushes buffered data to the destination  
D) Resets the stream to its initial state

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Immediately closes the stream and releases resources, optionally emitting an error event**

`stream.destroy([error])` destroys the stream immediately. It emits `'close'` and optionally `'error'` if an error is passed. No more data can be read or written. Use it when you need to abort a stream (e.g., user cancellation, timeout, or error recovery).
</details>

### MCQ 25
**What is the purpose of `express.urlencoded({ extended: true })`?**

A) Encodes URL parameters for safe transmission  
B) Parses query string parameters from the URL  
C) Parses URL-encoded form data. `extended: true` uses `qs` library for rich objects; `false` uses `querystring` for flat data  
D) Validates that the request URL is properly formatted

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Parses URL-encoded form data. `extended: true` uses `qs` library for rich objects; `false` uses `querystring` for flat data**

This middleware parses `Content-Type: application/x-www-form-urlencoded` bodies (e.g., HTML form submissions) and populates `req.body`. With `extended: true`, nested objects like `user[name]=John` become `{ user: { name: 'John' } }`. With `false`, you get flat key-value pairs.
</details>

### MCQ 26 — Output Prediction
**What is the output?**

```javascript
try {
    await Promise.reject(new Error('fail'));
} catch (e) {
    console.log('caught');
}
console.log('after');
```

A) fail, after  
B) caught, after  
C) Unhandled rejection (crashes)  
D) caught

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) caught, after**

When `await Promise.reject(...)` is used inside a try/catch, the rejection is caught by the catch block. The catch block logs `'caught'`, then execution continues to `console.log('after')`. This is one of the key benefits of async/await — synchronous-style error handling with try/catch.
</details>

In [ ]:
// Verify MCQ 26 (must be inside async function)
(async () => {
    try {
        await Promise.reject(new Error('fail'));
    } catch (e) {
        console.log('caught');
    }
    console.log('after');
})();

### MCQ 27
**What is `req.params` vs `req.query` vs `req.body` in Express?**

A) All three are the same, just different names  
B) params = URL path parameters (:id), query = query string (?key=val), body = request body (needs parser middleware)  
C) params = request body, query = URL params, body = query string  
D) They are used for different HTTP methods only

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) params = URL path parameters (:id), query = query string (?key=val), body = request body (needs parser middleware)**

- **req.params** — From route path: `/users/:id` → `req.params.id`
- **req.query** — From query string: `?page=2&sort=name` → `req.query.page`, `req.query.sort`
- **req.body** — From request body (JSON, form data). Requires `express.json()` or `express.urlencoded()` middleware.
</details>

### MCQ 28
**What is the difference between `.on('data')` and using `for await...of` to consume a readable stream?**

A) `.on('data')` is faster; `for await...of` is deprecated  
B) Both work; `for await...of` is the modern approach, handles backpressure and errors more cleanly  
C) `for await...of` only works with Transform streams  
D) `.on('data')` handles backpressure automatically; `for await...of` does not

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Both work; `for await...of` is the modern approach, handles backpressure and errors more cleanly**

Both consume readable streams. `for await...of` is preferred because: (1) it naturally handles backpressure (each iteration waits for the next chunk), (2) errors propagate via try/catch, (3) the loop exits cleanly on stream end. `.on('data')` is callback-based and requires manual backpressure handling (checking `readable.readableFlowing`).
</details>

### MCQ 29
**Which HTTP method(s) should be idempotent?**

A) GET, PUT, DELETE are idempotent; POST is NOT  
B) Only GET  
C) POST, PUT, PATCH  
D) All HTTP methods are idempotent

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) GET, PUT, DELETE are idempotent; POST is NOT**

**Idempotent** means making the same request multiple times has the same effect as making it once. GET (read), PUT (replace), DELETE (remove) are idempotent. POST creates a new resource each time — two identical POSTs create two resources, so it's NOT idempotent. PATCH is typically idempotent when applying the same partial update.
</details>

### MCQ 30
**What does `res.locals` do in Express?**

A) Stores session data across requests  
B) Caches response data for performance  
C) An object for passing data between middleware within the same request lifecycle  
D) Holds global application configuration

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) An object for passing data between middleware within the same request lifecycle**

`res.locals` is request-scoped. Middleware can set `res.locals.user = req.user`, and later middleware or route handlers can read it. Useful for passing auth user, flash messages, or computed values to templates. It does NOT persist across requests — each request gets a fresh `res.locals`.
</details>

### MCQ 31 — Output Prediction
**What is the output?**

```javascript
const p1 = Promise.resolve(1);
const p2 = Promise.resolve(2);
const p3 = Promise.reject('error');
Promise.all([p1, p2, p3])
    .then(r => console.log('then', r))
    .catch(e => console.log('catch', e));
```

A) catch error  
B) then [1, 2, 3]  
C) then [1, 2], catch error  
D) 1, 2, error

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) catch error**

`Promise.all()` fails fast — it rejects as soon as ANY promise in the array rejects. Since `p3` rejects, `Promise.all` rejects immediately with `'error'`. The `.then()` is skipped, and `.catch()` runs, logging `catch error`. The resolved values from p1 and p2 are discarded.
</details>

### MCQ 32
**What is the purpose of `app.route()` in Express?**

A) Defines the base URL for the application  
B) Create chainable route handlers for a single path to avoid duplicate path definitions  
C) Redirects requests to another route  
D) Registers a catch-all 404 handler

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Create chainable route handlers for a single path to avoid duplicate path definitions**

Instead of repeating the path in each method:
```javascript
app.get('/users/:id', ...);
app.put('/users/:id', ...);
app.delete('/users/:id', ...);
```
You can use:
```javascript
app.route('/users/:id')
    .get(...)
    .put(...)
    .delete(...);
```
</details>

### MCQ 33
**What does `stream.pipe()` return?**

A) The destination stream (allows chaining: readable.pipe(transform).pipe(writable))  
B) The source stream  
C) A Promise that resolves when piping completes  
D) undefined

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) The destination stream (allows chaining: readable.pipe(transform).pipe(writable))**

`pipe()` returns the destination stream, enabling chaining. Example: `fs.createReadStream('in.gz').pipe(zlib.createGunzip()).pipe(fs.createWriteStream('out.txt'))`. This pipes: file → decompress → file. Backpressure is automatically propagated through the chain.
</details>

### MCQ 34
**What is `queueMicrotask()` in Node.js?**

A) Schedules a callback to run after a specified delay  
B) Queues a microtask similar to Promise.then() but without creating a Promise  
C) Pauses the event loop until the microtask completes  
D) A deprecated API replaced by setImmediate()

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Queues a microtask similar to Promise.then() but without creating a Promise**

`queueMicrotask(callback)` schedules the callback to run in the microtask queue — the same queue used by Promise resolution. It runs after the current synchronous code and before the next macrotask (setTimeout, I/O). Useful when you need microtask timing without creating a Promise. Available globally in Node.js and browsers.
</details>

### MCQ 35
**Which Express middleware is used for serving static files?**

A) express.static()  
B) express.files()  
C) express.public()  
D) express.assets()

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) express.static()**

`express.static(root, [options])` serves static files (HTML, CSS, JS, images) from the specified root directory. Example: `app.use(express.static('public'))` serves files from the `public` folder at the root URL. For `GET /style.css`, it serves `public/style.css`.
</details>

### MCQ 36
**What is a Duplex stream? Give an example.**

A) A stream that is both readable and writable. Examples: TCP socket (net.Socket), WebSocket  
B) A stream that reads from two sources simultaneously  
C) A stream that transforms data in both directions  
D) A stream that can be paused and resumed

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) A stream that is both readable and writable. Examples: TCP socket (net.Socket), WebSocket**

Duplex streams implement both Readable and Writable interfaces. Data flows independently in both directions. Common examples: `net.Socket` (TCP), `tls.TLSSocket`, WebSocket connections. Transform streams are a type of Duplex where the writable side feeds the readable side (e.g., zlib, crypto).
</details>

### MCQ 37
**What happens if you call `res.json()` twice in Express?**

A) The second call overwrites the first response  
B) Both responses are sent; the client receives the last one  
C) Error — "Cannot set headers after they are sent to the client"  
D) Nothing; the second call is ignored

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Error — "Cannot set headers after they are sent to the client"**

Once you call `res.json()`, `res.send()`, or `res.end()`, the response headers are sent. You cannot modify headers or send another response. Calling `res.json()` again tries to set headers again — Node throws this error. Always ensure only one response is sent per request (use early returns, avoid code paths that could double-send).
</details>

### MCQ 38 — Output Prediction
**What is the output?**

```javascript
const delay = ms => new Promise(r => setTimeout(r, ms));
async function test() {
    const start = Date.now();
    const p1 = delay(100);
    const p2 = delay(100);
    await p1;
    await p2;
    console.log(Date.now() - start >= 100 && Date.now() - start < 200 ? 'parallel' : 'sequential');
}
test();
```

A) parallel  
B) sequential  
C) parallel (but prints ~200)  
D) Throws an error

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) parallel**

Both promises start immediately when created (`const p1 = delay(100)` and `const p2 = delay(100)`). They run in parallel. After ~100ms, both resolve. The two `await` statements run sequentially but both promises are already in flight, so total time is ~100ms. The condition `>= 100 && < 200` is true, so it logs `'parallel'`.
</details>

In [ ]:
// Verify MCQ 38
const delay = ms => new Promise(r => setTimeout(r, ms));
async function test() {
    const start = Date.now();
    const p1 = delay(100);
    const p2 = delay(100);
    await p1;
    await p2;
    console.log(Date.now() - start >= 100 && Date.now() - start < 200 ? 'parallel' : 'sequential');
}
test();

### MCQ 39
**What is the `next('route')` call in Express?**

A) Passes control to the next middleware in the stack  
B) Skips remaining handlers in the current route and moves to the next matching route  
C) Triggers the error handler with 'route' as the error  
D) Redirects the request to a different route

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Skips remaining handlers in the current route and moves to the next matching route**

When you call `next('route')` (must pass the string `'route'`), Express skips any remaining route handlers for the current path and tries the next route that matches. Useful when you have multiple handlers for the same path and want to skip to the next route based on a condition (e.g., "if not admin, skip to next GET handler").
</details>

### MCQ 40
**What is an AbortController and how does it relate to Promises?**

A) AbortController provides a signal to cancel async operations like fetch(), setTimeout (with signal option), and custom operations  
B) It aborts the entire Node.js process  
C) It replaces Promise.race() for timeout handling  
D) It is used only for Express request cancellation

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) AbortController provides a signal to cancel async operations like fetch(), setTimeout (with signal option), and custom operations**

`AbortController` creates an `AbortSignal` that can be passed to APIs that support cancellation. Call `controller.abort()` to signal cancellation. Supported in: `fetch(url, { signal })`, `setTimeout`/`setInterval` (with `timersPromises`), `EventTarget.addEventListener`, and custom async code that checks `signal.aborted`. Essential for canceling in-flight requests when the user navigates away or a component unmounts.
</details>

---
# PART B — Detailed Interview Q&A

---

### Q1: Explain the evolution from callbacks to Promises to async/await. Why did each one appear?

**Model Answer:**

**Callbacks** — The original Node.js async pattern. Functions take a callback `(err, result)` that fires when the operation completes.
- **Problem:** "Callback Hell" — deeply nested callbacks become unreadable, hard to debug, and error handling is repetitive.

**Promises (ES2015)** — Represent a future value. Enable chaining with `.then()/.catch()` and combinators like `Promise.all()`.
- **Solved:** Flat chaining instead of nesting. Better error propagation (one `.catch()` handles all errors in the chain).
- **Remaining problem:** Still requires `.then()` chaining; complex control flow (loops, conditionals) is still awkward.

**async/await (ES2017)** — Syntactic sugar over Promises. Makes async code look synchronous.
- **Solved:** Natural control flow (for loops, try/catch, if/else all work normally). Easier debugging (stack traces, breakpoints work as expected).
- **Important:** async/await IS Promises — it's not a different mechanism, just better syntax.

**In interviews, show this progression with a concrete example — reading a file, querying a DB, then processing the result — in all three styles.**

### Q2: What are Promise combinators and when would you use each one?

**Model Answer:**

| Combinator | Resolves when | Rejects when | Use case |
|-----------|-------------|-------------|----------|
| `Promise.all(promises)` | ALL fulfill | ANY rejects (fail-fast) | Fetch data from 3 APIs — all needed |
| `Promise.allSettled(promises)` | ALL settle | Never rejects | Send emails to 100 users — want results for each |
| `Promise.race(promises)` | FIRST settles | FIRST settles | Timeout: race API call vs timer |
| `Promise.any(promises)` | FIRST fulfills | ALL reject (AggregateError) | Try 3 CDNs — use whichever responds first |

**Practical examples:**

```javascript
// Timeout pattern with Promise.race
const result = await Promise.race([
    fetch('/api/data'),
    new Promise((_, reject) => setTimeout(() => reject(new Error('Timeout')), 5000))
]);

// Batch processing with partial failure tolerance
const results = await Promise.allSettled(users.map(u => sendEmail(u)));
const failed = results.filter(r => r.status === 'rejected');
console.log(`${failed.length} emails failed`);
```

### Q3: Why does `await` inside `forEach` not work as expected? How do you fix it?

**Model Answer:**

`Array.forEach()` calls each callback function but does NOT await the return value. It fires all callbacks immediately and returns `undefined`.

```javascript
// BROKEN — forEach doesn't wait
items.forEach(async (item) => {
    await processItem(item); // These all start at once!
});
console.log('Done'); // Runs before any item is processed!
```

**Fix for sequential processing:**
```javascript
for (const item of items) {
    await processItem(item); // Waits for each one
}
```

**Fix for parallel processing:**
```javascript
await Promise.all(items.map(item => processItem(item)));
```

**Fix for parallel with concurrency limit:**
```javascript
for (let i = 0; i < items.length; i += batchSize) {
    await Promise.all(items.slice(i, i + batchSize).map(processItem));
}
```

This is a very common interview question because it's a very common production bug.

### Q4: Describe the Express middleware execution flow. What happens when you call `next()`?

**Model Answer:**

Express processes requests through a **stack of middleware functions** in the order they're registered.

```
Request → [Logger] → [Auth] → [Validation] → [Route Handler] → Response
            next()     next()     next()         res.json()
```

**`next()` behavior:**
- `next()` — passes control to the next matching middleware/route
- `next(error)` — skips all remaining non-error middleware and jumps to the error handler
- Not calling `next()` or `res.send()` — the request **hangs** (client gets timeout)

**Key rules:**
1. Middleware executes in the order registered with `app.use()`
2. You can modify `req` and `res` objects (add properties, set headers)
3. A middleware can end the cycle by sending a response (then DON'T call `next()`)
4. Error middleware MUST have exactly 4 parameters and should be registered LAST
5. Route-specific middleware runs only for matching routes

### Q5: Design a RESTful API for a blog platform. Include routes, status codes, and response format.

**Model Answer:**

**Routes:**
```
POST   /api/v1/auth/register           → 201 { data: user }
POST   /api/v1/auth/login              → 200 { data: { token, user } }

GET    /api/v1/posts                   → 200 { data: [posts], meta: { page, total } }
GET    /api/v1/posts/:id               → 200 { data: post } | 404
POST   /api/v1/posts                   → 201 { data: post } | 400 | 401
PUT    /api/v1/posts/:id               → 200 { data: post } | 404 | 401 | 403
DELETE /api/v1/posts/:id               → 204 | 404 | 401 | 403

GET    /api/v1/posts/:id/comments      → 200 { data: [comments] }
POST   /api/v1/posts/:id/comments      → 201 { data: comment }

GET    /api/v1/users/:id               → 200 { data: user }
GET    /api/v1/users/:id/posts         → 200 { data: [posts] }
```

**Response format (consistent!):**
```json
{
    "status": "success",
    "data": { ... },
    "meta": { "page": 1, "limit": 20, "total": 150 }
}

{
    "status": "error",
    "message": "Post not found",
    "statusCode": 404
}
```

**Design principles applied:** nouns for resources, plural names, nested resources for relationships, versioned URL, query params for filtering (`?tag=nodejs&sort=-createdAt&page=2`), consistent error format.

### Q6: What is the `asyncHandler` pattern in Express and why do you need it?

**Model Answer:**

Express does NOT catch errors from rejected Promises automatically (until Express 5). If an async route handler throws, the error becomes an unhandled rejection.

**Without asyncHandler (verbose, repetitive):**
```javascript
app.get('/users/:id', async (req, res, next) => {
    try {
        const user = await User.findById(req.params.id);
        if (!user) throw new NotFoundError('User');
        res.json(user);
    } catch (err) {
        next(err); // Must manually forward to error handler
    }
});
```

**With asyncHandler (DRY):**
```javascript
const asyncHandler = (fn) => (req, res, next) =>
    Promise.resolve(fn(req, res, next)).catch(next);

app.get('/users/:id', asyncHandler(async (req, res) => {
    const user = await User.findById(req.params.id);
    if (!user) throw new NotFoundError('User');
    res.json(user);
}));
```

The wrapper catches any rejected Promise and forwards it to `next(err)`, which triggers the centralized error middleware. Libraries like `express-async-errors` do this globally.

### Q7: Explain how you would build a real-time notification system using EventEmitter.

**Model Answer:**

```javascript
const EventEmitter = require('events');

class NotificationService extends EventEmitter {
    notify(userId, event, data) {
        this.emit('notification', { userId, event, data, timestamp: Date.now() });
    }
}

const notifications = new NotificationService();

// Different services subscribe independently (decoupled!)
notifications.on('notification', async ({ userId, event, data }) => {
    await emailService.send(userId, event, data);
});

notifications.on('notification', async ({ userId, event, data }) => {
    await pushNotificationService.send(userId, event, data);
});

notifications.on('notification', ({ userId, event }) => {
    analyticsService.track(userId, event);
});

// Usage in your app:
notifications.notify(userId, 'order_placed', { orderId: '123' });
```

**Benefits:** decoupled architecture (adding a new notification channel requires zero changes to existing code), easy to test (mock the emitter), follows the Observer/Pub-Sub pattern.

**For production scale:** Replace the in-process EventEmitter with a message broker (Redis pub/sub, RabbitMQ) for cross-service communication.